# Neural networks on everyday problems, with Keras

Four small examples, one for each kind of answer a network can give. Each one follows the same five steps:

1. data
2. build the network
3. compile
4. fit (train)
5. predict

| Example | Question | Kind of answer | Output layer | Loss |
|---|---|---|---|---|
| 1 | How far does a car take to stop? | a number | 1 neuron, linear | `mse` |
| 2 | Will a student pass the exam? | yes / no | 1 neuron, sigmoid | `binary_crossentropy` |
| 3 | Apple, banana or orange? | one of 3, labels 0, 1, 2 | 3 neurons, softmax | `sparse_categorical_crossentropy` |
| 4 | T-shirt, jacket or raincoat? | one of 3, labels one-hot | 3 neurons, softmax | `categorical_crossentropy` |

All four datasets are made up for teaching, and each has only 9 to 12 rows.

In [1]:
import numpy as np
import keras

keras.utils.set_random_seed(42)   # same starting weights every run

## Example 1: Car stopping distance (a number)

Rule of thumb: distance (m) = reaction distance + braking distance

$$\text{distance} = \frac{\text{speed}}{10} \times 3 + \left(\frac{\text{speed}}{10}\right)^2$$

The made-up data below follows this rule exactly. Braking grows with speed **squared**, so this is a curve, not a straight line. One neuron can only draw a straight line, so we add a hidden layer.

In [2]:
# 1. Data
speed_kmh  = np.array([10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120], dtype=float)
distance_m = np.array([ 4, 10, 18, 28, 40, 54, 70, 88, 108, 130, 154, 180], dtype=float)

In [3]:
speed_scaled = speed_kmh / 100     # keep inputs small (0.1 to 1.2): helps training

# 2. Build the network
stopping_network = keras.Sequential([
    keras.Input(shape=(1,)),                     # 1 input: speed
    keras.layers.Dense(8, activation="relu"),    # hidden layer: lets it bend the line
    keras.layers.Dense(1, activation="linear"),  # output: distance in metres
])

In [4]:
# 3. Compile
stopping_network.compile(optimizer=keras.optimizers.Adam(learning_rate=0.05), loss="mse")

# 4. Fit (train)
stopping_network.fit(speed_scaled, distance_m, epochs=1000, verbose=0)   # with only 200 it can stop at a straight line

# 5. Predict
print("Example 1: Car stopping distance")
print("  speed    predicted    rule of thumb")
for speed in [25, 55, 85, 115]:
    predicted = stopping_network.predict(np.array([speed / 100]), verbose=0)[0][0]
    true = (speed / 10) * 3 + (speed / 10) ** 2
    print(f"  {speed:3} km/h   {predicted:6.1f} m      {true:6.1f} m")
print()


Example 1: Car stopping distance
  speed    predicted    rule of thumb
   25 km/h     13.4 m        13.8 m
   55 km/h     46.9 m        46.8 m
   85 km/h     99.1 m        97.8 m
  115 km/h    165.2 m       166.8 m



## Example 2: Will a student pass the exam? (yes / no)

- Inputs: hours studied, hours slept the night before
- Output: 1 = pass, 0 = fail

A **sigmoid** output squeezes the answer between 0 and 1, so we can read it as a probability of passing.

In [5]:
# 1. Data          studied  slept
student_hours = np.array([
                    [ 1,     4],
                    [ 2,     8],
                    [ 3,     5],
                    [ 4,     4],
                    [ 5,     8],
                    [ 6,     7],
                    [ 7,     5],
                    [ 8,     8],
                    [ 2,     6],
                    [ 9,     3],
                    [10,     7],
                    [ 6,     4],
], dtype=float)
passed = np.array([0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0], dtype=float)

hours_scaled = student_hours / 10     # keep inputs small

In [6]:
# 2. Build the network
exam_network = keras.Sequential([
    keras.Input(shape=(2,)),                      # 2 inputs: studied, slept
    keras.layers.Dense(4, activation="relu"),     # hidden layer
    keras.layers.Dense(1, activation="sigmoid"),  # output: probability of passing (0 to 1)
])

# 3. Compile  ("binary_crossentropy" is the usual loss for yes/no questions)
exam_network.compile(optimizer=keras.optimizers.Adam(learning_rate=0.05),
                     loss="binary_crossentropy", metrics=["accuracy"])

# 4. Fit (train)
exam_network.fit(hours_scaled, passed, epochs=500, verbose=0)

# 5. Predict
print("Example 2: Will a student pass?")
new_students = np.array([[1, 8], [5, 6], [9, 7], [7, 2]], dtype=float)
probabilities = exam_network.predict(new_students / 10, verbose=0)
for (studied, slept), p in zip(new_students, probabilities):
    answer = "PASS" if p[0] > 0.5 else "FAIL"
    print(f"  studied {studied:2.0f} h, slept {slept:.0f} h  ->  {p[0]:5.1%} chance  ->  {answer}")
print()

Example 2: Will a student pass?
  studied  1 h, slept 8 h  ->   0.2% chance  ->  FAIL
  studied  5 h, slept 6 h  ->  30.4% chance  ->  FAIL
  studied  9 h, slept 7 h  ->  100.0% chance  ->  PASS
  studied  7 h, slept 2 h  ->   0.2% chance  ->  FAIL



## Example 3: Apple, banana or orange? (one of 3)

- Inputs: weight (grams), length (cm)
- Output: 3 numbers, one probability per fruit, adding up to 100%

A **softmax** output does that sharing-out of probability. The answers are stored as whole numbers 0, 1, 2, so the loss is `sparse_categorical_crossentropy`.

In [7]:
fruit_names = ["apple", "banana", "orange"]

# 1. Data        weight  length
fruit_measurements = np.array([
                    [150,    7],    # apples
                    [170,    8],
                    [140,    7],
                    [120,   18],    # bananas: light for their length
                    [130,   20],
                    [110,   17],
                    [200,    9],    # oranges: heavier
                    [220,   10],
                    [190,    9],
], dtype=float)
fruit_type = np.array([0, 0, 0, 1, 1, 1, 2, 2, 2])   # 0 = apple, 1 = banana, 2 = orange

In [8]:
# Scale each column to roughly 0..1: weight / 250, length / 20
fruit_scaled = fruit_measurements / np.array([250, 20])

# 2. Build the network
fruit_network = keras.Sequential([
    keras.Input(shape=(2,)),                      # 2 inputs: weight, length
    keras.layers.Dense(8, activation="relu"),     # hidden layer
    keras.layers.Dense(3, activation="softmax"),  # 3 outputs: apple, banana, orange
])

# 3. Compile  ("sparse_categorical_crossentropy" = loss for picking one of several kinds,
#              when the answers are stored as 0, 1, 2)
fruit_network.compile(optimizer=keras.optimizers.Adam(learning_rate=0.05),
                      loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# 4. Fit (train)
fruit_network.fit(fruit_scaled, fruit_type, epochs=500, verbose=0)

# 5. Predict
print("Example 3: Which fruit is it?")
new_fruits = np.array([[160, 8], [125, 19], [210, 9]], dtype=float)
probabilities = fruit_network.predict(new_fruits / np.array([250, 20]), verbose=0)
for (weight, length), p in zip(new_fruits, probabilities):
    best = np.argmax(p)    # position of the highest probability
    print(f"  {weight:.0f} g, {length:.0f} cm  ->  {fruit_names[best]:6}   "
          f"(apple {p[0]:.0%}, banana {p[1]:.0%}, orange {p[2]:.0%})")

Example 3: Which fruit is it?
  160 g, 8 cm  ->  apple    (apple 100%, banana 0%, orange 0%)
  125 g, 19 cm  ->  banana   (apple 0%, banana 100%, orange 0%)
  210 g, 9 cm  ->  orange   (apple 0%, banana 0%, orange 100%)


## Example 4: What should I wear today? (one of 3, one-hot answers)

- Inputs: temperature (°C), rain (mm)
- Output: one probability each for t-shirt, jacket and raincoat

Same kind of question as the fruit, but the answers are written as **one-hot** lists, so the loss is `categorical_crossentropy`.

In [9]:
# What should I wear today?  T-shirt, jacket or raincoat?
# Uses loss="categorical_crossentropy", so the answers are written as ONE-HOT lists:
#   t-shirt  = [1, 0, 0]
#   jacket   = [0, 1, 0]
#   raincoat = [0, 0, 1]

import numpy as np
import keras

keras.utils.set_random_seed(42)   # same starting weights every run

clothes = ["t-shirt", "jacket", "raincoat"]

# 1. Data          temperature °C   rain mm
weather = np.array([
                    [28,            0],    # hot and dry      -> t-shirt
                    [25,            0],
                    [22,            1],
                    [12,            0],    # cold and dry     -> jacket
                    [ 8,            0],
                    [ 5,            1],
                    [18,           10],    # rainy            -> raincoat
                    [10,           15],
                    [24,            8],
], dtype=float)

# One-hot answers: a 1 in the position of the right item, 0 everywhere else
what_to_wear = np.array([
                    [1, 0, 0],   # t-shirt
                    [1, 0, 0],
                    [1, 0, 0],
                    [0, 1, 0],   # jacket
                    [0, 1, 0],
                    [0, 1, 0],
                    [0, 0, 1],   # raincoat
                    [0, 0, 1],
                    [0, 0, 1],
], dtype=float)



In [10]:
# Scale each column to roughly 0..1: temperature / 30, rain / 15
weather_scaled = weather / np.array([30, 15])

# 2. Build the network
wear_network = keras.Sequential([
    keras.Input(shape=(2,)),                      # 2 inputs: temperature, rain
    keras.layers.Dense(8, activation="relu"),     # hidden layer
    keras.layers.Dense(3, activation="softmax"),  # 3 outputs: t-shirt, jacket, raincoat
])

# 3. Compile: one-hot answers -> "categorical_crossentropy"
wear_network.compile(optimizer=keras.optimizers.Adam(learning_rate=0.05),
                     loss="categorical_crossentropy", metrics=["accuracy"])

# 4. Fit (train)
history = wear_network.fit(weather_scaled, what_to_wear, epochs=500, verbose=0)
print("final loss:", round(history.history["loss"][-1], 4))
print()

# 5. Predict for new days
new_days = np.array([[30, 0], [3, 0], [15, 12], [20, 0]], dtype=float)
probabilities = wear_network.predict(new_days / np.array([30, 15]), verbose=0)

for (temp, rain), p in zip(new_days, probabilities):
    best = np.argmax(p)    # position of the highest probability
    print(f"{temp:4.0f} °C, {rain:3.0f} mm rain  ->  {clothes[best]:8}   "
          f"(t-shirt {p[0]:.0%}, jacket {p[1]:.0%}, raincoat {p[2]:.0%})")

final loss: 0.0002

  30 °C,   0 mm rain  ->  t-shirt    (t-shirt 100%, jacket 0%, raincoat 0%)
   3 °C,   0 mm rain  ->  jacket     (t-shirt 0%, jacket 100%, raincoat 0%)
  15 °C,  12 mm rain  ->  raincoat   (t-shirt 0%, jacket 0%, raincoat 100%)
  20 °C,   0 mm rain  ->  t-shirt    (t-shirt 99%, jacket 1%, raincoat 0%)


## Read the results with care

- The networks often say 0% or 100%. With 9 to 12 made-up rows, a network can fit the training rows almost perfectly, and that makes it far too sure of itself. Real data is messier, and you would always keep a test set aside to check the model on rows it has never seen.
- Neural networks start from random weights. We fix the seed, so a re-run on the same computer gives the same numbers. On another computer the numbers can differ a little.